In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split

from config import (
    CNN_EPOCHS, CNN_BATCH_SIZE, CNN_VAL_SPLIT,
    N_CLASSES, CLASS_NAMES,
    SYNTHETIC_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR,
    RANDOM_SEED,
)
from preprocessing import preprocess_subject
from models.cnn import build_cnn
from evaluate import (
    compute_metrics,
    plot_confusion_matrix,
    plot_training_history,
    build_summary_table,
)

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

SUBJECT    = 1
DEMO_EPOCHS = 10 

print(f'TensorFlow {tf.__version__}')

## 1 · CNN Architecture (Fig. 6)

In [ ]:
model = build_cnn()
model.summary(line_length=80)

## 2 · Load Real + Synthetic Data

In [ ]:
X_real, y_real = preprocess_subject(SUBJECT, session='T')
print(f'Real      : {X_real.shape}  labels={np.unique(y_real, return_counts=True)[1]}')

syn_path = SYNTHETIC_DIR / f'synthetic_combined_s{SUBJECT:02d}.npz'
if not syn_path.exists():
    raise FileNotFoundError(
        f'{syn_path} not found — run notebook 02 or train_wgan.py first.'
    )
data  = np.load(str(syn_path))
X_syn = data['X'].astype(np.float32)
y_syn = data['y'].astype(np.int32)
print(f'Synthetic : {X_syn.shape}  labels={np.unique(y_syn, return_counts=True)[1]}')

X = np.concatenate([X_real, X_syn], axis=0)
y = np.concatenate([y_real, y_syn], axis=0)
print(f'Combined  : {X.shape}')

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=CNN_VAL_SPLIT,
    random_state=RANDOM_SEED, stratify=y
)
print(f'Train: {len(X_train)}  Val: {len(X_val)}')

In [ ]:
strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    model = build_cnn()

ckpt_path = MODELS_DIR / f'cnn_s{SUBJECT:02d}_best.weights.h5'
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(ckpt_path), monitor='val_accuracy',
        save_best_only=True, save_weights_only=True, verbose=0
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=DEMO_EPOCHS,
    batch_size=CNN_BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
plot_training_history(
    history.history,
    save_path=str(FIGURES_DIR / f'history_s{SUBJECT:02d}.png'),
    title_prefix=f'Subject {SUBJECT} — ',
)

In [ ]:
metrics = compute_metrics(model, X_val.astype(np.float32), y_val)

print(f"Accuracy  : {metrics['accuracy']:.4f}  ({metrics['accuracy']*100:.2f}%)")
print(f"Precision : {metrics['precision']:.4f}")
print(f"Recall    : {metrics['recall']:.4f}")
print(f"F1-Score  : {metrics['f1']:.4f}")
print(f"\nClassification Report:\n{metrics['report']}")

plot_confusion_matrix(
    metrics['cm'], CLASS_NAMES,
    save_path=str(FIGURES_DIR / f'cm_s{SUBJECT:02d}.png'),
    title=f'Subject {SUBJECT} — Normalised Confusion Matrix',
)

In [ ]:
CROSS_SUBJECT = 2

X_cross, y_cross = preprocess_subject(CROSS_SUBJECT, session='T')
cross_metrics = compute_metrics(model, X_cross.astype(np.float32), y_cross)

print(f"Cross-subject S{SUBJECT}→S{CROSS_SUBJECT}")
print(f"  Accuracy  : {cross_metrics['accuracy']:.4f}  ({cross_metrics['accuracy']*100:.2f}%)")
print(f"  F1-Score  : {cross_metrics['f1']:.4f}")

plot_confusion_matrix(
    cross_metrics['cm'], CLASS_NAMES,
    save_path=str(FIGURES_DIR / f'cm_s{SUBJECT:02d}_to_s{CROSS_SUBJECT:02d}.png'),
    title=f'S{SUBJECT}→S{CROSS_SUBJECT} Cross-Subject Confusion Matrix',
)

In [ ]:
import json

results = {
    'subject':   SUBJECT,
    'accuracy':  metrics['accuracy'],
    'precision': metrics['precision'],
    'recall':    metrics['recall'],
    'f1':        metrics['f1'],
    'cross_subject': CROSS_SUBJECT,
    'cross_accuracy': cross_metrics['accuracy'],
    'cross_f1':       cross_metrics['f1'],
}
with open(str(METRICS_DIR / f'cnn_metrics_s{SUBJECT:02d}.json'), 'w') as fh:
    json.dump(results, fh, indent=2)

df = build_summary_table()
if df is not None and not df.empty:
    display_cols = ['subject', 'accuracy', 'precision', 'recall', 'f1']
    print('\nAll-Subject Results (Table 4):')
    print(df[display_cols].to_string(index=False, float_format='{:.4f}'.format))
    print(f'\nMean accuracy: {df["accuracy"].mean()*100:.2f}%')

In [ ]:
RUN_ALL = False

if RUN_ALL:
    summary_rows = []
    for subj in range(1, 10):
        syn_path = SYNTHETIC_DIR / f'synthetic_combined_s{subj:02d}.npz'
        if not syn_path.exists():
            print(f'Subject {subj}: no synthetic data, skipping.')
            continue

        print(f'\n{'='*50}  Subject {subj}  {'='*50}')
        Xr, yr = preprocess_subject(subj, session='T')
        data   = np.load(str(syn_path))
        Xs     = data['X'].astype(np.float32)
        ys     = data['y'].astype(np.int32)
        X_all  = np.concatenate([Xr, Xs])
        y_all  = np.concatenate([yr, ys])

        Xtr, Xv, ytr, yv = train_test_split(
            X_all, y_all, test_size=CNN_VAL_SPLIT,
            random_state=RANDOM_SEED, stratify=y_all
        )
        with tf.distribute.MirroredStrategy().scope():
            m = build_cnn()
        m.fit(Xtr, ytr, validation_data=(Xv, yv),
              epochs=CNN_EPOCHS, batch_size=CNN_BATCH_SIZE, verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(
                  monitor='val_loss', patience=15, restore_best_weights=True)])

        met = compute_metrics(m, Xv, yv)
        summary_rows.append({'subject': subj, **{k: met[k]
                             for k in ['accuracy','precision','recall','f1']}})
        print(f'  Accuracy: {met["accuracy"]*100:.2f}%  F1: {met["f1"]:.4f}')

    df_all = pd.DataFrame(summary_rows)
    print('\n', df_all.to_string(index=False, float_format='{:.4f}'.format))